# CASAS Aruba — discovery su Colab ed esportazione locale

Questo notebook esegue la **discovery di modelli di processo PM4Py**; non addestra Gemini.

1. Carica questo notebook su [Google Colab](https://colab.research.google.com/), usando **File → Carica notebook**.
2. Carica il dataset con il browser in Google Drive, per esempio nella cartella `ArubaColab/input/`. Non è necessario caricare il dataset nella RAM del tuo PC con Python.
3. Scegli un runtime **CPU**. Se disponibile nel tuo piano, puoi scegliere RAM elevata; disponibilità e limiti sono variabili. La GPU non accelera automaticamente questi miner.
4. Esegui le celle in ordine. L'autorizzazione a Drive avviene nella terza cella di codice.

Il codice del progetto necessario è incorporato: non occorre clonare un repository né caricare chiavi API. Il file raw originale non viene incluso nell'archivio dei risultati.

**Limiti:** il preprocessing raw è streaming, ma la discovery richiede comunque che il log derivato e le strutture del miner stiano nella RAM di Colab. Più RAM non corregge mappatura, segmentazione o qualità metodologica. La modalità raw conserva la mappa e gli stati ON/OFF attuali; non estrae attività umane annotate.

Riferimenti: [FAQ Colab](https://research.google.com/colaboratory/faq.html), [file e Google Drive in Colab](https://colab.research.google.com/notebooks/io.ipynb).

Per confrontare algoritmi e soglie sullo stesso log usa il notebook separato `Aruba_Colab_Confronto_Modelli.ipynb`.


## 1. Installa le dipendenze di calcolo
Se Colab richiede un riavvio dopo l’installazione, riavvia il runtime e riparti dalla cella 2.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pm4py==2.7.23.11", "pandas>=2.0,<4", "numpy>=1.26,<3", "psutil"])


## 2. Prepara il codice incorporato

In [ ]:
from pathlib import Path
import sys
PROJECT = Path("/content/aruba_project")
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCES = {'colab_pipeline.py': '"""Strumenti Colab: ETL in streaming e modelli esportabili alla dashboard.\n\nNon modifica la pipeline locale né la semantica della mappa sensori corrente.\n"""\nimport hashlib\nimport importlib.metadata\nimport json\nimport pickle\nimport platform\nimport shutil\nfrom datetime import date, datetime\nfrom pathlib import Path\n\nimport pandas as pd\nimport pm4py\nfrom preprocessing import SENSOR_ACTIVITY_MAP, get_time_of_day\n\nCASE, ACTIVITY, TIME = "case:concept:name", "concept:name", "time:timestamp"\n\n\ndef preprocess_raw_to_csv(source, destination, start_date=None, end_date=None):\n    """Preprocessing del progetto in streaming: M/D, mappa, fasce e compressione.\n\n    Entrambi gli stati sono mantenuti. Il raw deve essere cronologico;\n    le date opzionali includono l\'intera giornata iniziale/finale.\n    """\n    import csv\n    from collections import Counter\n\n    source, destination = Path(source), Path(destination)\n    if source.resolve() == destination.resolve():\n        raise ValueError("Ingresso e uscita devono essere file diversi")\n    lower = date.fromisoformat(start_date) if start_date else None\n    upper = date.fromisoformat(end_date) if end_date else None\n    if lower and upper and lower > upper:\n        raise ValueError("START_DATE deve precedere o coincidere con END_DATE")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    counts = {\n        "lines": 0, "malformed_lines": 0, "excluded_non_motion_door": 0,\n        "excluded_unmapped": 0, "invalid_timestamps": 0, "outside_date_range": 0,\n        "mapped_events": 0, "written_events": 0,\n    }\n    unmapped, states, zones, periods = Counter(), Counter(), Counter(), Counter()\n    cases = set()\n    first_timestamp = final_timestamp = last_timestamp = None\n    previous = None\n    temporary = destination.with_suffix(".partial.csv")\n    try:\n        with source.open(encoding="utf-8-sig") as incoming, temporary.open("w", encoding="utf-8", newline="") as outgoing:\n            writer = csv.writer(outgoing)\n            writer.writerow([CASE, ACTIVITY, TIME])\n            for line in incoming:\n                counts["lines"] += 1\n                parts = line.split()\n                if len(parts) < 4:\n                    counts["malformed_lines"] += 1\n                    continue\n                if not parts[2].startswith(("M", "D")):\n                    counts["excluded_non_motion_door"] += 1\n                    continue\n                if parts[2] not in SENSOR_ACTIVITY_MAP:\n                    counts["excluded_unmapped"] += 1\n                    unmapped[parts[2]] += 1\n                    continue\n                try:\n                    timestamp = datetime.fromisoformat(parts[0] + " " + parts[1])\n                except ValueError:\n                    counts["invalid_timestamps"] += 1\n                    continue\n                if last_timestamp is not None and timestamp < last_timestamp:\n                    raise ValueError(f"Raw non cronologico alla riga {counts[\'lines\']}: ordinare il file prima della compressione streaming")\n                last_timestamp = timestamp\n                if (lower and timestamp.date() < lower) or (upper and timestamp.date() > upper):\n                    counts["outside_date_range"] += 1\n                    continue\n                first_timestamp = first_timestamp or timestamp\n                final_timestamp = timestamp\n                counts["mapped_events"] += 1\n                states[parts[3]] += 1\n                period = get_time_of_day(timestamp.hour)\n                case = f"{timestamp.date()}_{period}"\n                activity = SENSOR_ACTIVITY_MAP[parts[2]]\n                if (case, activity) != previous:\n                    writer.writerow([case, activity, timestamp.isoformat(sep=" ")])\n                    counts["written_events"] += 1\n                    zones[activity] += 1\n                    periods[period] += 1\n                    cases.add(case)\n                    previous = (case, activity)\n        if not counts["written_events"]:\n            raise ValueError(f"Nessun evento utilizzabile nel raw o nella finestra selezionata. Contatori: {counts}")\n        temporary.replace(destination)\n    except Exception:\n        temporary.unlink(missing_ok=True)\n        raise\n    counts.update({\n        "consecutive_duplicates_removed": counts["mapped_events"] - counts["written_events"],\n        "cases": len(cases), "zones": len(zones),\n        "first_input_event_in_window": first_timestamp.isoformat(sep=" "),\n        "last_input_event_in_window": final_timestamp.isoformat(sep=" "),\n        "requested_start_date": start_date, "requested_end_date": end_date,\n        "states_before_compression": dict(sorted(states.items())),\n        "unmapped_sensor_counts": dict(sorted(unmapped.items())),\n        "events_by_zone": dict(sorted(zones.items())),\n        "events_by_period": dict(sorted(periods.items())),\n        "sensor_activity_map": dict(SENSOR_ACTIVITY_MAP),\n        "case_definition": "date_plus_period_00_06_12_19_24",\n        "state_filter": "all_states_as_in_local_project",\n    })\n    return counts\n\n\ndef load_event_log(path):\n    # Carica solo le tre colonne necessarie. Il log derivato deve comunque stare in RAM.\n    df = pd.read_csv(path, usecols=[CASE, ACTIVITY, TIME], dtype={CASE: "string", ACTIVITY: "string"})\n    if df.empty or df[[CASE, ACTIVITY, TIME]].isna().any().any():\n        raise ValueError("Event log vuoto o con colonne PM4Py mancanti/nulle")\n    df[TIME] = pd.to_datetime(df[TIME], format="ISO8601", errors="raise")\n    df = df.sort_values(TIME, kind="stable").reset_index(drop=True)\n    # Condividi le stringhe ripetute senza utilizzare categorie incompatibili con alcune versioni PM4Py.\n    for col in (CASE, ACTIVITY):\n        codes, values = pd.factorize(df[col], sort=False)\n        df[col] = values.astype(object).take(codes)\n    return df[[CASE, ACTIVITY, TIME]]\n\n\ndef discover_one(df, miner="Inductive Miner", noise_threshold=0.2, dependency_threshold=0.5):\n    common = dict(case_id_key=CASE, activity_key=ACTIVITY, timestamp_key=TIME)\n    if miner == "Inductive Miner":\n        model = pm4py.discover_petri_net_inductive(df, noise_threshold=noise_threshold, **common)\n    elif miner == "Heuristic Miner":\n        model = pm4py.discover_petri_net_heuristics(df, dependency_threshold=dependency_threshold, **common)\n    elif miner == "Alpha Miner":\n        model = pm4py.discover_petri_net_alpha(df, **common)\n    else:\n        raise ValueError(f"Miner non supportato: {miner}")\n    return {miner: model}\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as source:\n        for block in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef export_results(df, models, directory, source_path, settings, scores=None, evaluation_metadata=None):\n    root = Path(directory)\n    data = root / "data"\n    data.mkdir(parents=True, exist_ok=True)\n    df[[CASE, ACTIVITY, TIME]].to_csv(data / "dataset_aruba_attivita.csv", index=False)\n    with (data / "models.pkl").open("wb") as handle:\n        pickle.dump(models, handle, protocol=pickle.HIGHEST_PROTOCOL)\n    (data / "metrics.json").write_text(json.dumps(scores or {}, indent=2), encoding="utf-8")\n    if evaluation_metadata is not None:\n        target = data / "evaluation_metadata.json"\n        if Path(evaluation_metadata).resolve() != target.resolve():\n            shutil.copyfile(evaluation_metadata, target)\n    else:\n        (data / "evaluation_metadata.json").unlink(missing_ok=True)\n    for i, model in enumerate(models.values(), start=1):\n        pm4py.write_pnml(*model, str(data / f"model_{i}.pnml"))\n    versions = {name: importlib.metadata.version(name) for name in ["pm4py", "pandas", "numpy"]}\n    manifest = {\n        "python": platform.python_version(), "packages": versions,\n        "source_name": Path(source_path).name, "source_sha256": sha256_file(source_path),\n        "events": len(df), "cases": int(df[CASE].nunique()), "activities": int(df[ACTIVITY].nunique()),\n        "settings": settings, "evaluation": "completed" if scores is not None else "not_run",\n        "files_sha256": {str(p.relative_to(root)): sha256_file(p) for p in data.iterdir() if p.is_file()},\n    }\n    (root / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    (root / "requirements_colab.txt").write_text(\n        "".join(f"{name}=={version}\\n" for name, version in versions.items()), encoding="utf-8")\n    return root\n\n\ndef package_results(directory):\n    directory = Path(directory)\n    return Path(shutil.make_archive(str(directory), "zip", root_dir=directory))\n', 'preprocessing.py': 'import pandas as pd\nimport os\nimport pm4py\n\n# ==========================================\n# CONFIGURAZIONE COSTANTI\n# ==========================================\nCASE_ID_KEY = "case:concept:name"\nACTIVITY_KEY = "concept:name"\nTIMESTAMP_KEY = "time:timestamp"\n\n# ==========================================\n# 1. MAPPA ATTIVITÀ\n# ==========================================\nSENSOR_ACTIVITY_MAP = {\n    # ZONA NOTTE\n    "M001": "Camera_Letto", "M002": "Camera_Letto", \n    "M003": "Camera_Letto", "M004": "Camera_Letto",\n    "M005": "Bagno_Principale", "M006": "Bagno_Principale", "M007": "Bagno_Principale",\n    \n    # ZONA GIORNO\n    "M008": "Soggiorno", "M009": "Soggiorno", "M010": "Soggiorno", \n    "M011": "Soggiorno", "M012": "Soggiorno", "M013": "Soggiorno", \n    "M014": "Sala_Pranzo",\n    \n    # CUCINA\n    "M015": "Cucina_Prep", "M016": "Cucina_Prep", "M017": "Cucina_Prep", \n    "M018": "Cucina_Prep", "M019": "Cucina_Prep",\n    \n    # UFFICIO\n    "M020": "Ufficio", "M021": "Ufficio", "M022": "Ufficio",\n    \n    # CORRIDOI\n    "M023": "Corridoio", "M024": "Corridoio",\n    \n    # SECONDA ZONA\n    "M025": "Camera_2", "M026": "Bagno_2",\n    "M027": "Camera_2", "M028": "Bagno_2", \n    "M029": "Camera_2", "M030": "Camera_2", "M031": "Camera_2",\n    \n    # PORTE\n    "D001": "Uscita_Principale", "D002": "Uscita_Retro", "D004": "Garage"\n}\n\ndef get_time_of_day(hour):\n    if 0 <= hour < 6: return "Notte"\n    elif 6 <= hour < 12: return "Mattina"\n    elif 12 <= hour < 19: return "Pomeriggio"\n    else: return "Sera"\n\n# ==========================================\n# 2. CARICAMENTO E PULIZIA\n# ==========================================\ndef load_and_transform_dataset(file_path):\n    print(f"1. Caricamento file: {file_path}")\n    \n    if not os.path.exists(file_path):\n        print(f"ERRORE: File \'{file_path}\' mancante.")\n        # Invece di sys.exit, solleviamo un\'eccezione per essere più pythonic in un modulo\n        raise FileNotFoundError(f"File \'{file_path}\' mancante.")\n\n    data = []\n    try:\n        with open(file_path, \'r\') as f:\n            for line in f:\n                parts = line.strip().split()\n                if len(parts) >= 4:\n                    data.append(parts[:4])\n    except Exception as e:\n        print(f"Errore lettura: {e}")\n        raise e\n\n    df = pd.DataFrame(data, columns=["date", "time", "sensor_id", "value"])\n\n    # Manteniamo entrambi gli stati ON/OFF e OPEN/CLOSE per compatibilità\n    # con il dataset derivato; le etichette identificano zone, non azioni validate.\n    # Filtro Sensori (Solo Motion e Door, NO Temperatura)\n    print("   Filtro Sensori (Eliminazione Temperatura)...")\n    df = df[df[\'sensor_id\'].str.startswith((\'M\', \'D\'))].copy()\n\n    # Timestamp\n    df[\'temp_ts\'] = df[\'date\'] + \' \' + df[\'time\']\n    df[TIMESTAMP_KEY] = pd.to_datetime(df[\'temp_ts\'], format="%Y-%m-%d %H:%M:%S.%f", errors=\'coerce\')\n    df = df.dropna(subset=[TIMESTAMP_KEY]).sort_values(TIMESTAMP_KEY, kind="stable")\n\n    # Mappatura Attività\n    print("   Conversione Sensori -> Attività Umane...")\n    df[\'activity_raw\'] = df[\'sensor_id\'].map(SENSOR_ACTIVITY_MAP)\n    df[\'activity_raw\'] = df[\'activity_raw\'].fillna("Altro")\n    df = df[df[\'activity_raw\'] != "Altro"].copy()\n\n    # Suddivisione Temporale\n    df[\'hour\'] = df[TIMESTAMP_KEY].dt.hour\n    df[\'period\'] = df[\'hour\'].apply(get_time_of_day)\n    df[CASE_ID_KEY] = df[TIMESTAMP_KEY].dt.date.astype(str) + "_" + df[\'period\']\n\n    # Compressione (Rimuove duplicati consecutivi)\n    print(f"   Eventi grezzi: {len(df)}")\n    mask = (df[\'activity_raw\'] != df[\'activity_raw\'].shift(1)) | \\\n           (df[CASE_ID_KEY] != df[CASE_ID_KEY].shift(1))\n    df = df[mask].copy()\n    \n    df[ACTIVITY_KEY] = df[\'activity_raw\']\n    # Mantieni colonne utili\n    final_df = df[[CASE_ID_KEY, ACTIVITY_KEY, TIMESTAMP_KEY]].reset_index(drop=True)\n    \n    print(f"   Eventi finali ottimizzati: {len(final_df)}")\n    return final_df\n\ndef extract_process_structure(df):\n    """\n    Estrae il DFG e le varianti.\n    """\n    if df is None or df.empty:\n        return "Nessun dato disponibile."\n    try:\n        # 1. DFG\n        dfg, start_activities, end_activities = pm4py.discover_dfg(df)\n        sorted_dfg = sorted(dfg.items(), key=lambda x: x[1], reverse=True)\n\n        structure_str = "--- STRUTTURA DEL PROCESSO (DFG) ---\\n"\n        structure_str += f"Start: {list(start_activities.keys())}\\nEnd: {list(end_activities.keys())}\\n"\n        structure_str += "FLUSSI PRINCIPALI (A -> B [Freq]):\\n"\n        for (act_from, act_to), count in sorted_dfg[:30]:\n            structure_str += f"- {act_from} -> {act_to} [{count}]\\n"\n\n        # 2. Varianti (con fix robustezza)\n        variants = pm4py.get_variants_as_tuples(df)\n\n        def get_count(val):\n            return val if isinstance(val, int) else len(val)\n\n        sorted_variants = sorted(variants.items(), key=lambda x: get_count(x[1]), reverse=True)\n\n        structure_str += "\\n--- VARIANTI TOP 5 ---\\n"\n        for path, val in sorted_variants[:5]:\n            count = get_count(val)\n            path_str = " -> ".join(path[:20])\n            if len(path) > 20:\n                path_str += f" -> ... ({len(path)} eventi nella traccia completa)"\n            structure_str += f"- ({count} casi): {path_str}\\n"\n\n        return structure_str\n    except Exception as e:\n        return f"Errore estrazione struttura: {str(e)}"\n', 'evaluation.py': 'import json\nimport math\nfrom hashlib import sha256\nimport numpy as np\nimport pm4py\n# Costanti PM4Py\nCASE_ID_KEY = "case:concept:name"\nACTIVITY_KEY = "concept:name"\nTIMESTAMP_KEY = "time:timestamp"\nRANDOM_SEED = 42\nMAX_EVAL_CASES = 50  # Numero di periodi da valutare (Safe Mode)\n\ndef evaluate(df, net, im, fm):\n    """\n    Valuta un singolo modello su un DataFrame (già convertito in log internamente da PM4Py se necessario).\n    """\n    try:\n        # Replay token-based (Il migliore per performance)\n        # Nota: PM4Py accetta DataFrame direttamente in molte funzioni recenti, \n        # ma assicuriamoci di passare le chiavi corrette.\n        fitness = pm4py.fitness_token_based_replay(\n            df, net, im, fm, \n            case_id_key=CASE_ID_KEY, \n            activity_key=ACTIVITY_KEY, \n            timestamp_key=TIMESTAMP_KEY\n        )\n        precision = pm4py.precision_token_based_replay(\n            df, net, im, fm, \n            case_id_key=CASE_ID_KEY, \n            activity_key=ACTIVITY_KEY, \n            timestamp_key=TIMESTAMP_KEY\n        )\n        # Generalization e Simplicity potrebbero essere pesanti, li includiamo se richiesto o li omettiamo per velocità\n        # Per ora manteniamo Fitness e Precision che sono i più critici.\n        \n        return {\n            "fitness": fitness[\'log_fitness\'], \n            "precision": precision\n        }\n    except Exception as e:\n        return {"error": str(e)}\n\ndef evaluate_models_safe(df, models, metadata_path=None):\n    print("\\n3. Valutazione (Safe Mode)...")\n    unique_cases = df[CASE_ID_KEY].unique()\n    \n    if len(unique_cases) > MAX_EVAL_CASES:\n        sampled_cases = np.random.default_rng(RANDOM_SEED).choice(unique_cases, MAX_EVAL_CASES, replace=False)\n        eval_df = df[df[CASE_ID_KEY].isin(sampled_cases)].copy()\n        print(f"   Valutazione su campione di {MAX_EVAL_CASES} periodi.")\n    else:\n        eval_df = df\n        print(f"   Valutazione su tutti i {len(unique_cases)} periodi.")\n\n    if metadata_path is not None:\n        import pandas as pd\n        fingerprint_df = df[[CASE_ID_KEY, ACTIVITY_KEY, TIMESTAMP_KEY]].copy()\n        fingerprint_df[TIMESTAMP_KEY] = pd.to_datetime(fingerprint_df[TIMESTAMP_KEY]).astype(str)\n        metadata = {\n            "seed": RANDOM_SEED,\n            "max_eval_cases": MAX_EVAL_CASES,\n            "total_cases": int(len(unique_cases)),\n            "evaluated_cases": [str(c) for c in eval_df[CASE_ID_KEY].unique()],\n            "evaluated_events": int(len(eval_df)),\n            "dataset_sha256": sha256(fingerprint_df.to_csv(index=False).encode("utf-8")).hexdigest(),\n            "pm4py_version": pm4py.__version__,\n            "selection_criterion": "harmonic_mean_fitness_precision",\n            "scope": "in_sample: modelli scoperti sull\'intero log, valutati su un sottoinsieme"\n        }\n\n    scores = {}\n    for name, model in models.items():\n        if isinstance(model, dict) and "error" in model:\n            scores[name] = model\n            continue\n        \n        print(f"   Valutando {name}...", end=" ")\n        try:\n            net, im, fm = model\n            res = evaluate(eval_df, net, im, fm)\n            scores[name] = res\n            \n            if "error" not in res:\n                print(f"-> OK (F: {res[\'fitness\']:.2f}, P: {res[\'precision\']:.2f})")\n            else:\n                print(f"-> FALLITO ({res[\'error\']})")\n                \n        except Exception as e:\n            print(f"-> FALLITO (Eccezione imprevista)")\n            scores[name] = {"error": str(e)}\n            \n    if metadata_path is not None:\n        metadata["metrics_sha256"] = sha256(json.dumps(scores, sort_keys=True).encode("utf-8")).hexdigest()\n        with open(metadata_path, "w", encoding="utf-8") as handle:\n            json.dump(metadata, handle, indent=2, ensure_ascii=False)\n    return scores\n\ndef f1_score(metrics):\n    """Media armonica di fitness e precision; rifiuta metriche incomplete/non valide."""\n    if not isinstance(metrics, dict) or "error" in metrics:\n        return None\n    try:\n        fitness = float(metrics["fitness"])\n        precision = float(metrics["precision"])\n    except (KeyError, TypeError, ValueError):\n        return None\n    if not all(math.isfinite(v) and 0 <= v <= 1 for v in (fitness, precision)):\n        return None\n    return 2 * fitness * precision / (fitness + precision) if fitness + precision else 0.0\n\n\ndef choose_best(results: dict):\n    """Seleziona il modello con la maggiore media armonica fitness/precision."""\n    valid = {name: m for name, m in results.items() if f1_score(m) is not None}\n    if not valid:\n        return None, None\n    name = max(valid, key=lambda key: f1_score(valid[key]))\n    return name, valid[name]\n\n\ndef choose_best_model_internal(scores):\n    """Alias compatibile per la dashboard; stesso criterio di choose_best."""\n    return choose_best(scores)[0]\n', 'project_paths.py': '"""Percorsi indipendenti dalla directory di avvio."""\nfrom pathlib import Path\n\nPROJECT_DIR = Path(__file__).resolve().parent\nDATA_DIR = PROJECT_DIR / "data"\nCONFIG_PATH = PROJECT_DIR / "config.json"\nDATASET_PATH = DATA_DIR / "aruba.20111101-20111128_20260218.180945.txt"\nOUTPUT_CSV = DATA_DIR / "dataset_aruba_attivita.csv"\nOUTPUT_MODELS = DATA_DIR / "models.pkl"\nOUTPUT_METRICS = DATA_DIR / "metrics.json"\nEVALUATION_METADATA = DATA_DIR / "evaluation_metadata.json"\n'}
for name, content in SOURCES.items():
    (PROJECT / name).write_text(content, encoding="utf-8")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("Codice disponibile in", PROJECT)


## 3. Collega Drive e configura il file di ingresso
Modifica **DATASET** con il percorso esatto del tuo file. Puoi copiare il percorso dal pannello File di Colab dopo aver collegato Drive.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
import json
import shutil
import gc
import psutil
import pm4py
from colab_pipeline import (preprocess_raw_to_csv, load_event_log, discover_one,
                           export_results, package_results)

drive.mount("/content/drive")

DATASET = Path("/content/drive/MyDrive/ArubaColab/input/aruba.txt")  # MODIFICA QUI
INPUT_KIND = "raw_sensors"  # raw nuovo: lascia questo valore
# Finestra facoltativa: None elabora tutto il file, date YYYY-MM-DD incluse.
START_DATE = None
END_DATE = None
MINER = "Inductive Miner"  # alternative: "Heuristic Miner", "Alpha Miner"
NOISE_THRESHOLD = 0.2  # parametro esistente; non è stato ottimizzato

# Solo se abiliti la valutazione facoltativa nella sezione 8.
MAX_EVAL_CASES = 20

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = Path("/content/aruba_runs") / RUN_ID
DRIVE_DIR = Path("/content/drive/MyDrive/ArubaColab/results") / RUN_ID
PREPARED_CSV = RUN_DIR / "data/dataset_aruba_attivita.csv"
RUN_DIR.mkdir(parents=True, exist_ok=False)
PREPARED_CSV.parent.mkdir(exist_ok=True)

if not DATASET.is_file():
    raise FileNotFoundError(f"File non trovato: {DATASET}. Modifica DATASET e riesegui la cella.")
print("RAM totale del runtime:", round(psutil.virtual_memory().total / 2**30, 2), "GiB")
print("Output persistente:", DRIVE_DIR)


def backup_to_drive():
    archive = package_results(RUN_DIR)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RUN_DIR, DRIVE_DIR, dirs_exist_ok=True)
    saved_zip = DRIVE_DIR / "aruba_results.zip"
    shutil.copy2(archive, saved_zip)
    print("Backup completato:", saved_zip)
    return archive


## 4. Preprocessing del nuovo dataset raw — stessa logica del progetto

Prima di eseguire questa sezione **decomprimi l'eventuale archivio scaricato** e imposta `DATASET` sul file testuale CASAS. Lascia `INPUT_KIND="raw_sensors"`.

La trasformazione riproduce il progetto:
1. Legge data, ora, sensore e stato dalle prime quattro colonne.
2. Conserva movimento `M` e porte `D`; esclude temperatura e sensori non mappati, riportandone i conteggi.
3. Costruisce i timestamp ed esclude quelli non validi.
4. Usa la stessa `SENSOR_ACTIVITY_MAP` del progetto.
5. Crea casi **data + fascia**: Notte 00–06, Mattina 06–12, Pomeriggio 12–19, Sera 19–24.
6. Rimuove le ripetizioni consecutive della stessa zona all'interno dello stesso caso.
7. Salva le colonne PM4Py `case:concept:name`, `concept:name`, `time:timestamp`.

Restano inclusi entrambi gli stati ON/OFF e OPEN/CLOSE. Le eventuali annotazioni aggiuntive del dataset non vengono convertite in attività: le etichette rimangono quelle della mappa sensori del progetto, la cui validità va verificata per la versione del nuovo dataset.

**Memoria:** il raw è letto riga per riga e l'output è scritto direttamente su disco, senza caricarlo tutto in RAM. A differenza dell'ordinamento in memoria della pipeline locale, qui il file deve essere già cronologico: in caso contrario viene segnalata la riga e la procedura si interrompe. Il file di output precedente non viene sostituito da un risultato parziale.

`START_DATE` e `END_DATE` sono facoltativi e inclusivi, nella sola modalità raw. Con entrambi a `None` viene elaborato l'intero dataset, senza campionamento. Il report con filtri, periodo e conteggi viene salvato su Drive prima di caricare il log in RAM.

Per un CSV **già elaborato**, imposta invece `INPUT_KIND="event_log_csv"`: questa sezione lo copia senza riapplicare la mappa; la sezione 5 ne verifica le colonne.


In [ ]:
import pandas as pd
from preprocessing import SENSOR_ACTIVITY_MAP

# Mostra la mappa effettivamente applicata, identica al progetto locale.
if INPUT_KIND == "raw_sensors":
    display(pd.DataFrame(sorted(SENSOR_ACTIVITY_MAP.items()), columns=["Sensore", "Zona del progetto"]))
    etl_info = preprocess_raw_to_csv(
        DATASET, PREPARED_CSV, start_date=START_DATE, end_date=END_DATE)
    print("Righe lette:", f"{etl_info['lines']:,}")
    print("Eventi mappati prima della compressione:", f"{etl_info['mapped_events']:,}")
    print("Duplicati consecutivi rimossi:", f"{etl_info['consecutive_duplicates_removed']:,}")
    print("Eventi finali:", f"{etl_info['written_events']:,}")
    print("Casi:", etl_info["cases"], "— Zone:", etl_info["zones"])
    print("Periodo degli eventi raw selezionati:", etl_info["first_input_event_in_window"],
          "→", etl_info["last_input_event_in_window"])
    exclusions = {key: etl_info[key] for key in ["malformed_lines", "excluded_non_motion_door",
                 "excluded_unmapped", "invalid_timestamps", "outside_date_range"]}
    print("Esclusioni:", exclusions)
    if etl_info["unmapped_sensor_counts"]:
        print("ATTENZIONE — sensori M/D non mappati:", etl_info["unmapped_sensor_counts"])
    display(pd.DataFrame(list(etl_info["events_by_period"].items()), columns=["Fascia", "Eventi compressi"]))
elif INPUT_KIND == "event_log_csv":
    if START_DATE or END_DATE:
        raise ValueError("START_DATE/END_DATE sono disponibili solo per raw_sensors; filtra prima il CSV elaborato")
    shutil.copyfile(DATASET, PREPARED_CSV)
    etl_info = {"mode": "preformatted_event_log", "preprocessing_applied": False}
    print("CSV già elaborato copiato; la mappa sensori non viene riapplicata.")
else:
    raise ValueError("INPUT_KIND non riconosciuto")

(RUN_DIR / "preprocessing_report.json").write_text(
    json.dumps(etl_info, indent=2, ensure_ascii=False), encoding="utf-8")
# Backup prima del caricamento del CSV derivato in RAM.
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_DIR, DRIVE_DIR, dirs_exist_ok=True)
print("CSV e report salvati su Drive:", DRIVE_DIR)


## 5. Carica e verifica il log derivato

Ora viene caricato in RAM solo il CSV già filtrato e compresso. Controlla il riepilogo prima di proseguire: la discovery e la valutazione successive useranno **questo `df`**, non il file raw originale.

Se il runtime si interrompe qui o durante la discovery, il CSV e `preprocessing_report.json` restano su Drive. Puoi riutilizzare quel CSV in una nuova esecuzione con `INPUT_KIND="event_log_csv"`.


In [ ]:
df = load_event_log(PREPARED_CSV)
print(f"Log per la discovery: {len(df):,} eventi; {df['case:concept:name'].nunique():,} casi; {df['concept:name'].nunique()} zone")
print("Timestamp CSV:", df["time:timestamp"].min(), "→", df["time:timestamp"].max())
print("Stima deep del DataFrame:", round(df.memory_usage(deep=True).sum() / 2**30, 3), "GiB")
print("RAM ancora disponibile:", round(psutil.virtual_memory().available / 2**30, 2), "GiB")
display(df.head(20))


## 6. Discovery di un modello alla volta

La GPU non è utilizzata dai miner qui chiamati. Inizia con Inductive; Alpha su log grandi o complessi può essere molto costoso. Questa cella usa l'intero log preparato, senza campionamento nascosto.

Se il runtime esaurisce la RAM anche qui, riparti dal CSV preparato già salvato su Drive (`INPUT_KIND="event_log_csv"`): servirà più RAM oppure un esperimento su un sottoinsieme temporale dichiarato. Abbassare la dimensione dei blocchi di lettura non riduce la memoria intrinseca della discovery.


In [ ]:
gc.collect()
models = discover_one(df, miner=MINER, noise_threshold=NOISE_THRESHOLD)
print("Modelli creati:", list(models))


## 7. Esporta subito e salva su Drive
Questa cella crea un archivio già utilizzabile nella dashboard. Le metriche rimangono vuote finché non esegui la valutazione facoltativa.

In [ ]:
settings = {"input_kind": INPUT_KIND, "miner": MINER,
            "noise_threshold": NOISE_THRESHOLD if MINER == "Inductive Miner" else None,
            "dependency_threshold": 0.5 if MINER == "Heuristic Miner" else None,
            "etl": etl_info, "discovery_scope": "entire_prepared_log"}
export_results(df, models, RUN_DIR, DATASET, settings)
archive = backup_to_drive()


## 8. Valutazione facoltativa, dopo il backup

Fitness e soprattutto precision possono richiedere molta RAM e tempo. Per scaricare subito il modello puoi **saltare questa cella** e passare alla sezione 9. Per eseguirla imposta `RUN_EVALUATION=True`.

Il campione di al massimo 20 casi completi è usato soltanto per la valutazione; la discovery resta sull'intero log. Seed e casi selezionati sono salvati. È una valutazione **in-sample**, non una verifica su giorni nuovi. Un runtime interrotto durante questa fase lascia disponibile il backup della sezione 7.


In [ ]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    import evaluation
    evaluation.MAX_EVAL_CASES = MAX_EVAL_CASES
    metadata_path = RUN_DIR / "data/evaluation_metadata.json"
    scores = evaluation.evaluate_models_safe(df, models, metadata_path=metadata_path)
    export_results(df, models, RUN_DIR, DATASET, settings, scores=scores,
                   evaluation_metadata=metadata_path)
    archive = backup_to_drive()
    print(json.dumps(scores, indent=2))
else:
    print("Valutazione saltata. Il modello e il CSV sono già esportati.")


## 9. Scarica il risultato sul PC
Se il download nel browser fallisce o il file è molto grande, scarica `aruba_results.zip` direttamente dalla cartella di Drive stampata sopra.

In [ ]:
from google.colab import files
files.download(str(archive))


## 10. Usa il modello nella dashboard locale

1. Estrai `aruba_results.zip` in una nuova cartella del PC.
2. Avvia la dashboard locale e usa gli upload manuali di CSV, PKL e JSON dalla cartella `data/` dell'archivio. Se avevi già caricato dati, ricarica la pagina per vedere gli upload. In alternativa, dopo aver salvato una copia dei vecchi risultati, copia i file estratti dalla cartella `data/` in quella del progetto e premi **Ricarica risultati**. Non serve trasferire il raw.
3. Se hai saltato la valutazione, `metrics.json` è vuoto: è previsto. Il modello è comunque visualizzabile.
4. L'archivio contiene anche PNML per i modelli, `run_manifest.json` con versioni/impronte/provenienza e `requirements_colab.txt` con le versioni di calcolo. In caso di incompatibilità di caricamento usa un ambiente locale con queste versioni più i requisiti della dashboard. Il formato PNML offre un'esportazione aggiuntiva indipendente dal pickle Python.

Il PC non esegue più la discovery, ma deve ancora caricare il CSV e renderizzare il modello: un event log molto grande può richiedere RAM anche durante la consultazione. Il trasferimento su Colab risolve dove avviene il calcolo, non garantisce risultati scientificamente migliori o RAM illimitata.
